# Assignment 2: Gradio dashboard

Bottle cap defect inspection. Run via `uv run python main.py app`.

Everything here comes from what the training stage already produced: the committed
weights in `weights/` and the files in `results/`. Nothing is trained at startup.
Styling and structure follow the Assignment 1 dashboard.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

ROOT = Path.cwd()
RESULTS = ROOT / "results"

## Load what training produced

In [ ]:
model = YOLO(str(ROOT / "weights" / "best.pt"))

operating_point = json.loads((RESULTS / "operating_point.json").read_text())
metrics = json.loads((RESULTS / "metrics.json").read_text())
dataset_stats = json.loads((RESULTS / "dataset_stats.json").read_text())
scores = {s: pd.read_csv(RESULTS / f"scores_{s}.csv") for s in ("valid", "test")}

COST = operating_point["cost_matrix_eur"]
C_FN, C_FP = COST["false_negative"], COST["false_positive"]
C_TP, C_TN, C_REVIEW = COST["true_positive"], COST["true_negative"], COST["human_review"]
BASE_RATE = operating_point["base_rate"]
CHOSEN_T = operating_point["threshold"]
CHOSEN_LO = operating_point["band"]["pass_below"]
CHOSEN_HI = operating_point["band"]["fail_at_or_above"]

GOOD = metrics["good_class"]
IMGSZ = metrics["imgsz"]
CLASS_NAMES = metrics["box"]["confusion_matrix"]["labels"][:-1]
GOOD_IDX = CLASS_NAMES.index(GOOD)
THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)

## Cost model (same definitions as the training notebook)

In [ ]:
def rates(y, s, lo, hi):
    """Share of defective / good bottles that end up passed, reviewed, failed."""
    fail, review = s >= hi, (s >= lo) & (s < hi)
    pos, neg = max(int(y.sum()), 1), max(int((~y).sum()), 1)
    return {
        "tp": (fail & y).sum() / pos, "rev_pos": (review & y).sum() / pos,
        "fn": 1 - ((fail | review) & y).sum() / pos,
        "fp": (fail & ~y).sum() / neg, "rev_neg": (review & ~y).sum() / neg,
        "tn": 1 - ((fail | review) & ~y).sum() / neg,
    }


def cost_per_1000(y, s, lo, hi, base_rate):
    """Expected EUR per 1,000 bottles. lo == hi means no review band."""
    r = rates(y, s, lo, hi)
    defective = r["tp"] * C_TP + r["rev_pos"] * (C_REVIEW + C_TP) + r["fn"] * C_FN
    good = r["fp"] * C_FP + r["rev_neg"] * (C_REVIEW + C_TN) + r["tn"] * C_TN
    return 1000 * (base_rate * defective + (1 - base_rate) * good)


def review_share(y, s, lo, hi, base_rate):
    r = rates(y, s, lo, hi)
    return base_rate * r["rev_pos"] + (1 - base_rate) * r["rev_neg"]


def arrays(split):
    df = scores[split]
    return df.defective.to_numpy(dtype=bool), df.score.to_numpy()


def ordered(lo, hi):
    return (lo, hi) if lo <= hi else (hi, lo)

## Look and feel

In [ ]:
PRIMARY = "#1b4b82"        # dark blue
PRIMARY_DARK = "#0b2545"   # navy
PRIMARY_LIGHT = "#e3effa"  # light blue tint
SECONDARY = "#2a9d8f"      # teal green
ACCENT = "#5aa9e6"         # light blue
PASS = "#2e9e5b"           # green
FAIL = "#d64545"           # red
REVIEW = ACCENT
INK = "#33465c"
INK_MUTED = "#6b7a8c"
GRID = "#dde6ee"
TEXT = "#14213d"
BG = "#f4f9fd"
TINT = "#f1f7fd"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.edgecolor": INK_MUTED, "axes.labelcolor": INK,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.titlecolor": TEXT, "axes.titleweight": "bold", "axes.titlesize": 12,
    "legend.frameon": False,
})


def card(body, title="", accent=PRIMARY, tinted=False):
    background = TINT if tinted else "#ffffff"
    heading = f'<div class="card-title" style="color:{accent}">{title}</div>' if title else ""
    return (f'<div class="card" style="border-left:4px solid {accent};background:{background}">'
            f'{heading}<div class="card-body">{body}</div></div>')


def tab_header(title, body):
    return card(f"<p>{body}</p>", title=title)


def takeaway(title, body):
    return card(f"<p>{body}</p>", title=title, accent=PRIMARY_DARK, tinted=True)


def kpi_tile(name, value, label, color=PRIMARY, big=True):
    size = "kpi-value" if big else "kpi-value-sm"
    return (f'<div class="kpi" style="border-top:4px solid {color}">'
            f'<div class="kpi-name" style="color:{color}">{name}</div>'
            f'<div class="{size}">{value}</div><div class="kpi-label">{label}</div></div>')

## Plots

In [ ]:
def confusion_plot(lo, hi, split):
    """Bottle-level confusion matrix: actual (rows) against pass / review / fail (columns)."""
    y, s = arrays(split)
    cols = [s < lo, (s >= lo) & (s < hi), s >= hi]
    counts = np.array([[int((c & ~y).sum()) for c in cols], [int((c & y).sum()) for c in cols]])
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.imshow(counts, cmap="Blues", vmin=0, vmax=max(counts.max(), 1))
    ax.grid(False)
    for i in range(2):
        for j in range(3):
            ax.text(j, i, counts[i, j], ha="center", va="center", fontsize=18, fontweight="bold",
                    color="white" if counts[i, j] > counts.max() / 2 else TEXT)
    ax.set_xticks(range(3), ["Pass", "Human review", "Fail"])
    ax.set_yticks(range(2), ["Actually good", "Actually defective"])
    ax.set_title(f"Bottles in the {split} split (n={len(y)})")
    fig.tight_layout()
    return fig


def strategy_plot(lo, hi, base_rate):
    y, s = arrays("test")
    values = {
        "Do nothing": 1000 * (base_rate * C_FN + (1 - base_rate) * C_TN),
        "A person checks everything": 1000 * (base_rate * (C_REVIEW + C_TP) + (1 - base_rate) * (C_REVIEW + C_TN)),
        "Model decides alone (chosen threshold)": cost_per_1000(y, s, CHOSEN_T, CHOSEN_T, base_rate),
        "Model + human review (current band)": cost_per_1000(y, s, lo, hi, base_rate),
    }
    fig, ax = plt.subplots(figsize=(6, 3.2))
    bars = ax.barh(list(values), list(values.values()),
                   color=[INK_MUTED, SECONDARY, ACCENT, PRIMARY])
    ax.bar_label(bars, labels=[f"€{v:,.0f}" for v in values.values()], padding=4, fontsize=9)
    ax.set_xlabel("Expected cost per 1,000 bottles (€)")
    ax.set_xlim(0, max(values.values()) * 1.18)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    return fig


def cost_curve_plot(base_rate):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    for split, color in (("valid", SECONDARY), ("test", PRIMARY)):
        y, s = arrays(split)
        curve = [cost_per_1000(y, s, t, t, base_rate) for t in THRESHOLDS]
        ax.plot(THRESHOLDS, curve, color=color, linewidth=2,
                label=f"{split} split" + (" (threshold chosen here)" if split == "valid" else ""))
        if split == "test":
            at_chosen = cost_per_1000(y, s, CHOSEN_T, CHOSEN_T, base_rate)
    ax.axvline(CHOSEN_T, color=TEXT, linestyle="--", linewidth=1)
    ax.scatter([CHOSEN_T], [at_chosen], color=TEXT, zorder=5, s=40,
               label=f"chosen threshold {CHOSEN_T:.2f}: €{at_chosen:,.0f} per 1,000 (test)")
    ax.set_xlabel("Confidence threshold (flag a bottle as defective at or above this score)")
    ax.set_ylabel("Expected cost per 1,000 bottles (€)")
    ax.set_title(f"Cost against threshold at a {base_rate:.1%} defect rate")
    ax.legend(fontsize=8, loc="upper center")
    fig.tight_layout()
    return fig


def per_class_bars():
    table = pd.DataFrame(metrics["box"]["per_class"]).T
    fig, ax = plt.subplots(figsize=(8, 4.2))
    x = np.arange(len(table))
    for k, (col, label, color) in enumerate((("precision", "Precision", PRIMARY),
                                             ("recall", "Recall", SECONDARY),
                                             ("ap50", "AP50", ACCENT))):
        ax.bar(x + (k - 1) * 0.27, table[col], 0.27, label=label, color=color)
    ax.set_xticks(x, [f"{n}\n(n={int(c)})" for n, c in zip(table.index, table.test_instances)], fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title("Per-class results on the test split (n = test instances)")
    ax.legend(fontsize=8, ncol=3, loc="lower left")
    fig.tight_layout()
    return fig


def box_confusion_plot():
    cm = metrics["box"]["confusion_matrix"]
    matrix = np.array(cm["matrix"]).T  # stored [predicted][true]; show true on rows
    fig, ax = plt.subplots(figsize=(6.4, 5.2))
    ax.imshow(matrix, cmap="Blues")
    ax.grid(False)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if matrix[i, j]:
                ax.text(j, i, int(matrix[i, j]), ha="center", va="center", fontsize=10,
                        color="white" if matrix[i, j] > matrix.max() / 2 else TEXT)
    ax.set_xticks(range(len(cm["labels"])), cm["labels"], rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(cm["labels"])), cm["labels"], fontsize=8)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Detection confusion matrix, test split (conf >= {cm.get('conf', 0.001):g})")
    fig.tight_layout()
    return fig

## Dashboard callbacks

In [ ]:
def operating_view(lo, hi, base_rate, per_day, seconds_per_check, minutes_per_person):
    lo, hi = ordered(lo, hi)
    y, s = arrays("test")
    r = rates(y, s, lo, hi)
    share = review_share(y, s, lo, hi, base_rate)
    cost = cost_per_1000(y, s, lo, hi, base_rate)
    reviewed_per_day = per_day * share
    people = reviewed_per_day * seconds_per_check / 60 / max(minutes_per_person, 1)
    tiles = "".join([
        kpi_tile("Expected cost", f"€{cost:,.0f}", "per 1,000 bottles"),
        kpi_tile("Sent to a person", f"{share:.1%}", f"{share * 1000:,.0f} per 1,000 bottles", SECONDARY),
        kpi_tile("Checks per day", f"{reviewed_per_day:,.0f}", f"at {per_day:,.0f} bottles a day", ACCENT),
        kpi_tile("People needed", f"{people:.2f}", "full-time inspectors", INK),
    ])
    missed = 1000 * base_rate * r["fn"]
    rejected = 1000 * (1 - base_rate) * r["fp"]
    note = card(
        f"At these thresholds, per 1,000 bottles: <b>{missed:.1f}</b> defective bottles are passed "
        f"unnoticed and <b>{rejected:.1f}</b> good bottles are rejected without a check. "
        f"Scores below <b>{lo:.2f}</b> pass, scores from <b>{lo:.2f}</b> up to <b>{hi:.2f}</b> go to a "
        f"person, and scores of <b>{hi:.2f}</b> or more fail. "
        f"Counts are measured on the {len(y)} test bottles and re-weighted to a {base_rate:.1%} defect rate.",
        accent=INK,
    )
    return (confusion_plot(lo, hi, "test"), strategy_plot(lo, hi, base_rate),
            f'<div class="kpi-row">{tiles}</div>{note}')


def predict_image(image, lo, hi):
    if image is None:
        return None, card("Upload a photo of a bottle cap to see the model's verdict.", title="No image yet")
    lo, hi = ordered(lo, hi)
    frame = np.array(image.convert("RGB"))
    everything = model.predict(frame, conf=0.001, imgsz=IMGSZ, verbose=False)[0]
    shown = model.predict(frame, conf=max(lo, 0.001), imgsz=IMGSZ, verbose=False)[0]
    cls = everything.boxes.cls.cpu().numpy().astype(int)
    conf = everything.boxes.conf.cpu().numpy()
    defect = cls != GOOD_IDX
    score = float(conf[defect].max()) if defect.any() else 0.0
    if score >= hi:
        verdict, accent = "FAIL: defective", FAIL
    elif score >= lo:
        verdict, accent = "SEND FOR HUMAN REVIEW", REVIEW
    else:
        verdict, accent = "PASS", PASS
    rows = "".join(
        f"<li>{CLASS_NAMES[int(c)]}: {p:.2f}</li>"
        for c, p in zip(shown.boxes.cls.cpu().numpy(), shown.boxes.conf.cpu().numpy())
    ) or "<li>No detections at this threshold</li>"
    body = (f"<p>Defect score <b>{score:.2f}</b> (the highest confidence among defect classes). "
            f"Detections at or above {max(lo, 0.001):.2f}:</p><ul>{rows}</ul>")
    return shown.plot()[:, :, ::-1], card(body, title=verdict, accent=accent, tinted=True)


def class_tables():
    per_class = pd.DataFrame(metrics["box"]["per_class"]).T
    per_class.insert(0, "class", per_class.index)
    per_class = per_class.rename(columns={
        "precision": "Precision", "recall": "Recall", "ap50": "AP50",
        "ap50_95": "mAP50-95", "test_instances": "Test instances"}).round(3)
    counts = pd.DataFrame({
        split: {n: d["classes"][n]["instances"] for n in CLASS_NAMES}
        for split, d in dataset_stats.items()
    })
    counts.insert(0, "class", counts.index)
    bottle = pd.DataFrame({k: v for k, v in metrics["bottle_level_test"].items() if isinstance(v, dict)}).T
    bottle = bottle[["tp", "fp", "fn", "tn", "accuracy", "precision", "recall"]].round(3)
    bottle.insert(0, "model", bottle.index)
    return per_class, counts, bottle


def overall_cards():
    box = metrics["box"]
    return "".join([
        kpi_tile("mAP50 (all classes)", f"{box['map50']:.3f}", "test split, detection", PRIMARY),
        kpi_tile("mAP50-95", f"{box['map50_95']:.3f}", "test split, detection", SECONDARY),
        kpi_tile("Chosen threshold", f"{CHOSEN_T:.2f}", "picked on the validation split", ACCENT),
        kpi_tile("Model", metrics["model"].replace(".pt", ""), f"{metrics['epochs']} epochs, {IMGSZ}px", INK, big=False),
    ])

## Theme and styles

In [ ]:
THEME_COLORS = {
    "body_background_fill": BG,
    "body_text_color": TEXT,
    "body_text_color_subdued": INK_MUTED,
    "background_fill_primary": "#ffffff",
    "background_fill_secondary": PRIMARY_LIGHT,
    "block_background_fill": "#ffffff",
    "block_border_color": GRID,
    "block_label_background_fill": "#ffffff",
    "block_label_text_color": INK,
    "block_title_text_color": INK,
    "border_color_primary": GRID,
    "input_background_fill": "#ffffff",
    "input_border_color": "#dddddd",
    "color_accent_soft": PRIMARY_LIGHT,
}
THEME = gr.themes.Base(
    primary_hue=gr.themes.colors.blue,
    neutral_hue=gr.themes.colors.gray,
    font=[gr.themes.GoogleFont("Nunito"), "Helvetica Neue", "Arial", "sans-serif"],
    radius_size=gr.themes.sizes.radius_lg,
).set(
    **THEME_COLORS,
    **{f"{name}_dark": value for name, value in THEME_COLORS.items()},
    color_accent=PRIMARY,
    block_shadow="0 1px 4px rgba(0,0,0,.06)",
)

CSS = f"""
body, .gradio-container {{ background-color: {BG} !important; }}
.gradio-container {{ max-width: 100% !important; width: 100% !important; }}

.hero {{
    background: linear-gradient(135deg, {PRIMARY_DARK} 0%, {PRIMARY} 100%);
    border-radius: 16px; padding: 28px 32px;
}}
.hero h1 {{ margin: 0 0 8px 0; font-size: 26px; font-weight: 800; color: #ffffff; }}
.hero p  {{ margin: 0; font-size: 14px; line-height: 1.6; color: #ffffff; }}

.tab-container {{ gap: 8px; height: auto !important; padding: 6px 0; }}
.tab-container::after {{ display: none !important; }}
.tab-container button {{
    background: #ffffff !important; color: {INK} !important;
    border: 1px solid #e5e5e5 !important; border-radius: 999px !important;
    padding: 8px 20px !important; font-weight: 700 !important;
    font-size: 14px !important; height: auto !important;
}}
.tab-container button:hover:not(.selected) {{
    background: {PRIMARY_LIGHT} !important; color: {PRIMARY_DARK} !important;
    border-color: {PRIMARY} !important;
}}
.tab-container button.selected {{
    background: {PRIMARY} !important; color: #ffffff !important; border-color: {PRIMARY} !important;
}}
.tab-container button.selected::after {{ display: none !important; }}

.card {{
    border-radius: 12px; padding: 16px 20px; margin: 4px 0;
    border: 1px solid {GRID}; box-shadow: 0 1px 4px rgba(0,0,0,.06);
}}
.card-title {{ font-size: 15px; font-weight: 800; margin-bottom: 6px; }}
.card-body, .card-body p, .card-body li {{ font-size: 13.5px; line-height: 1.7; color: {TEXT}; margin: 0 0 8px 0; }}
.card-body p:last-child {{ margin-bottom: 0; }}
.card-body b {{ color: #111111; }}

.kpi-row {{ display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 10px; }}
.kpi {{
    flex: 1; min-width: 200px; position: relative;
    background: #ffffff; border-radius: 12px; padding: 18px 22px;
    border: 1px solid {GRID}; box-shadow: 0 1px 4px rgba(0,0,0,.06);
}}
.kpi-name  {{ font-size: 12px; font-weight: 800; text-transform: uppercase;
              letter-spacing: .06em; margin-bottom: 10px; }}
.kpi-value {{ font-size: 30px; font-weight: 800; color: {TEXT}; line-height: 1; }}
.kpi-value-sm {{ font-size: 20px; font-weight: 700; color: {TEXT}; line-height: 1.2; }}
.kpi-label {{ font-size: 11px; font-weight: 700; color: {INK_MUTED}; text-transform: uppercase;
              letter-spacing: .05em; margin-top: 4px; }}

.section-title {{ font-size: 12px; font-weight: 800; color: {INK}; text-transform: uppercase;
                  letter-spacing: .08em; margin: 6px 0 2px 2px; }}
"""

FORCE_LIGHT_JS = """
() => {
  const url = new URL(window.location.href);
  if (url.searchParams.get('__theme') !== 'light') {
    url.searchParams.set('__theme', 'light');
    window.location.replace(url.href);
  }
}
"""

## Layout

In [ ]:
with gr.Blocks(title="Bottle Defect Inspection", fill_width=True) as demo:
    gr.HTML("""
    <div class="hero">
      <h1>Bottle cap defect inspection &mdash; operating-point dashboard</h1>
      <p>Every bottle is passed, failed, or sent to a person. The two thresholds below decide which,
         and the euro cost of being wrong decides where they should sit.</p>
    </div>""")

    with gr.Tab("Pass / review / fail"):
        gr.HTML(tab_header(
            "Move the two thresholds and watch the cost change",
            "A bottle's defect score is the highest confidence among its defect-class detections. Scores "
            "below the lower threshold pass, scores between the two go to a person, and scores at or above "
            "the upper threshold fail. The numbers below are measured on the held-out test split.",
        ))
        with gr.Row():
            lo_slider = gr.Slider(0.01, 0.99, value=CHOSEN_LO, step=0.01, label="Pass below (lower threshold)")
            hi_slider = gr.Slider(0.01, 0.99, value=CHOSEN_HI, step=0.01, label="Fail at or above (upper threshold)")
        with gr.Row():
            base_slider = gr.Slider(0.005, 0.30, value=BASE_RATE, step=0.005,
                                    label="Assumed share of defective bottles in production")
            per_day = gr.Number(value=20000, label="Bottles inspected per day (assumption)", precision=0)
            seconds = gr.Number(value=15, label="Seconds per human check (assumption)", precision=0)
            shift = gr.Number(value=420, label="Inspection minutes per person per day (assumption)", precision=0)
        kpis = gr.HTML()
        with gr.Row(equal_height=True):
            with gr.Column():
                gr.HTML('<div class="section-title">Confusion matrix</div>')
                confusion = gr.Plot(show_label=False)
            with gr.Column():
                gr.HTML('<div class="section-title">Priced against the alternatives</div>')
                strategies = gr.Plot(show_label=False)
        controls = [lo_slider, hi_slider, base_slider, per_day, seconds, shift]
        for control in controls:
            control.change(operating_view, controls, [confusion, strategies, kpis])
        demo.load(operating_view, controls, [confusion, strategies, kpis])

    with gr.Tab("Cost against threshold"):
        gr.HTML(tab_header(
            "Where does the sweep bottom out?",
            "Expected cost per 1,000 bottles at every confidence threshold, with the threshold chosen on "
            "the validation split marked. Change the assumed defect rate to see how fragile that choice is.",
        ))
        curve_base = gr.Slider(0.005, 0.30, value=BASE_RATE, step=0.005,
                               label="Assumed share of defective bottles in production")
        curve = gr.Plot(show_label=False)
        curve_base.change(cost_curve_plot, curve_base, curve)
        demo.load(cost_curve_plot, curve_base, curve)
        gr.HTML(takeaway(
            "How to read it",
            f"Costs: a missed defect €{C_FN:g}, a rejected good bottle €{C_FP:g}, a human check "
            f"€{C_REVIEW:g}. The validation curve picked the threshold; the test curve is what that "
            "choice would have cost on bottles the choice never saw. A wide gap between the two means "
            "the threshold is noisy.",
        ))

    with gr.Tab("Per-class results"):
        gr.HTML(tab_header(
            "Where the model is strong and where it breaks",
            "Detection results on the test split, per class. An overall average hides the weak classes, "
            "so check the instance counts too: a class with few test instances gives a number you cannot "
            "trust to the decimal.",
        ))
        gr.HTML(f'<div class="kpi-row">{overall_cards()}</div>')
        per_class_df, counts_df, bottle_df = class_tables()
        with gr.Row(equal_height=True):
            with gr.Column():
                gr.Plot(per_class_bars(), show_label=False)
            with gr.Column():
                gr.Plot(box_confusion_plot(), show_label=False)
        gr.HTML('<div class="section-title">Per-class numbers (test split)</div>')
        gr.Dataframe(per_class_df, interactive=False)
        gr.HTML('<div class="section-title">Instances per class and split</div>')
        gr.Dataframe(counts_df, interactive=False)
        gr.HTML('<div class="section-title">Bottle-level comparison with the baselines (test split)</div>')
        gr.Dataframe(bottle_df, interactive=False)

    with gr.Tab("Try an image"):
        gr.HTML(tab_header(
            "Upload a bottle photo",
            "Uses the two thresholds from the first tab: the lower one is also the lowest confidence "
            "at which a box is drawn.",
        ))
        with gr.Row(equal_height=True):
            upload = gr.Image(type="pil", label="Bottle photo")
            annotated = gr.Image(label="Model predictions", interactive=False)
        verdict = gr.HTML()
        for trigger in (upload.change, lo_slider.change, hi_slider.change):
            trigger(predict_image, [upload, lo_slider, hi_slider], [annotated, verdict])

demo.launch(theme=THEME, css=CSS, js=FORCE_LIGHT_JS)